In [9]:

import pandas as pd
import os

# "/mnt/c/Git_Repo/mohenjo-daro/data/20250312_MJD_processed_data_reclassified_2.csv"
df_path= '/Users/saraeichner/Desktop/github/mohenjo-daro/data/20260209_MJD_processed_data.csv'
#df = pd.read_csv(df)
df = pd.read_csv(df_path, encoding="mac_roman")
df = df[['Plate','Block','House','Room']]
df.head()

,Plate,Block,House,Room
0,037-00b-X,7,4,50
1,054-005-X,9,10,55
2,054-012-X,7,4,72
3,054-014-X,9,8,15
4,055-001-X,9,4,7


In [ ]:
image_folders = os.listdir('./archieve')
image_folders

images = []
for (root,dirs,files) in os.walk('archieve', topdown=False ):
    if root != 'archieve':
        files_long = [ [f"{root}/{i}",i] for i in files]
        images.extend( files_long )

# sample of images
images = pd.DataFrame(images, columns=['Path', 'ImageName'])
images['Plate'] = images['ImageName'].str.split(".").str[0]
images['Plate'] = images['Plate'].str.upper()

images.head()

,Path,ImageName,Plate
0,archieve/images_1/052-001-x.png,052-001-x.png,052-001-X
1,archieve/images_1/052-002-x.png,052-002-x.png,052-002-X
2,archieve/images_1/052-003-x.png,052-003-x.png,052-003-X
3,archieve/images_1/052-004-x.png,052-004-x.png,052-004-X
4,archieve/images_1/052-005-x.png,052-005-x.png,052-005-X


In [27]:
df_img = df.join( images.set_index('Plate'), on='Plate', how='left', rsuffix='_img' )
df_img = df_img.dropna(subset=['Plate','Path'], axis=0 )
print(f"Total number of images matched: {df_img.shape[0]}")
df_img.head()

Total number of images matched: 492


,Plate,Block,House,Room,Path,ImageName
0,054-005-X,9,10,55,archieve/images_1/054-005-x.png,054-005-x.png
1,054-012-X,7,4,72,archieve/images_1/054-012-x.png,054-012-x.png
1,054-012-X,7,4,72,archieve/images_b7-h3-4-5/054-012-x.png,054-012-x.png
2,054-014-X,9,8,15,archieve/images_1/054-014-x.png,054-014-x.png
3,055-001-X,9,4,7,archieve/images_1/055-001-x.png,055-001-x.png


In [29]:
df_img = df_img.fillna( {
    'Block': '0',
    'House': '0',
    'Room': '0',
    'Path': '0'
} )

df_img.drop_duplicates(subset=['Block', 'House', 'Room'], keep='first' )

,Plate,Block,House,Room,Path,ImageName
0,054-005-X,9,10,55,archieve/images_1/054-005-x.png,054-005-x.png
1,054-012-X,7,4,72,archieve/images_1/054-012-x.png,054-012-x.png
2,054-014-X,9,8,15,archieve/images_1/054-014-x.png,054-014-x.png
3,055-001-X,9,4,7,archieve/images_1/055-001-x.png,055-001-x.png
5,055-007-X,7,1,9,archieve/images_1/055-007-x.png,055-007-x.png
...,...,...,...,...,...,...
841,132-006-X,7,3,3,archieve/images_2/132-006-x.png,132-006-x.png
984,140-044-X,7,5,62,archieve/images_b7-h3-4-5/140-044-x.png,140-044-x.png
986,140-048-X,7,4,0,archieve/images_b7-h3-4-5/140-048-x.png,140-048-x.png
989,140-055-X,7,5,63,archieve/images_b7-h3-4-5/140-055-x.png,140-055-x.png


In [43]:
values = df_img[["Block","House","Room"]].drop_duplicates()[["Block","House","Room"]].values

for block, house, room in values:
    # make dirs
    block_dir = f"./sorted/Block_{block}"
    house_dir = f"./sorted/Block_{block}/House_{house}"
    room_dir = f"./sorted/Block_{block}/House_{house}/Room_{room}"
    
    os.makedirs(block_dir, exist_ok=True)
    os.makedirs(house_dir, exist_ok=True)
    os.makedirs(room_dir, exist_ok=True)


In [47]:
for i,r in df_img.iterrows():
    image_path = r['Path']
    destination = f"./sorted/Block_{r['Block']}/House_{r['House']}/Room_{r['Room']}"

    if os.path.exists(f"{destination}/{os.path.basename(image_path)}"):
        print(f"Image already exists in the destination: {destination}")
        continue
    else:
        os.system(f'cp "{image_path}" "{destination}"')


Image already exists in the destination: ./sorted/Block_9/House_10/Room_55
Image already exists in the destination: ./sorted/Block_7/House_4/Room_72
Image already exists in the destination: ./sorted/Block_7/House_3/Room_44
Image already exists in the destination: ./sorted/Block_7/House_5/Room_64
Image already exists in the destination: ./sorted/Block_7/House_3/Room_0
Image already exists in the destination: ./sorted/Block_7/House_3/Room_0
Image already exists in the destination: ./sorted/Block_7/House_3/Room_0
Image already exists in the destination: ./sorted/Block_7/House_3/Room_40
Image already exists in the destination: ./sorted/Block_7/House_3/Room_44
Image already exists in the destination: ./sorted/Block_7/House_5/Room_53
Image already exists in the destination: ./sorted/Block_7/House_5/Room_64
Image already exists in the destination: ./sorted/Block_7/House_3/Room_52
Image already exists in the destination: ./sorted/Block_7/House_3/Room_52
Image already exists in the destination:

In [69]:
new_images = []
for (root,dirs,files) in os.walk('./sorted', topdown=False):
    if len(files) > 0:


        files_long = [ (file,os.path.join(root, file)) for file in files ]
        new_images.extend( files_long )
new_images = pd.DataFrame(new_images, columns=['ImageName', 'Path'])
new_images['ImageName'] = new_images['ImageName'].str.split(".").str[0].str.upper()

new_images
        

,ImageName,Path
0,059-011-X,./sorted/Block_10/House_0/Room_0/059-011-x.png
1,075-018-X,./sorted/Block_10/House_0/Room_0/075-018-x.png
2,067-022-X,./sorted/Block_10/House_1/Room_11/067-022-x.png
3,076-013-X,./sorted/Block_10/House_1/Room_11/076-013-x.png
4,066-012-X,./sorted/Block_10/House_1/Room_2/066-012-x.png
...,...,...
425,057-009-X,./sorted/Block_9/House_9/Room_65/057-009-x.png
426,077-019-X,./sorted/Block_9/House_9/Room_65/077-019-x.png
427,077-009-X,./sorted/Block_9/House_9/Room_66/077-009-x.png
428,056-052-X,./sorted/Block_9/House_9/Room_68/056-052-x.png


In [ ]:
# "/mnt/c/Git_Repo/mohenjo-daro/data/20250312_MJD_processed_data_reclassified_2.csv"
df = '/Users/saraeichner/Desktop/github/mohenjo-daro/data/20250407_MJD_processed_data.csv'
df = pd.read_csv(df)
df = df.drop(columns = ['photo','photo_link','photo_shortor'])

delete = [i for i in df.columns if "Unname" in i]
df = df.drop(columns=delete, errors='ignore')

df.head()

,Plate,Block,House,Room,Level_ft,Time_Cat,Feature,Type,Text,N1,Class,x,y,Class_new
0,054-005-X,9,10,55,-9.1,Late III,Pottery - Incised and Painted - Upper,NaN,NaN,Artefacts,Domestic,68.137116,27.327116,Feeding / holding
1,054-012-X,7,4,72,-7.7,Late III,Tools,Candle-stick,Pottery candle-stick. Made of the usual pink w...,Artefacts,Craft,68.137494,27.326809,Making / Craft / Food
2,054-014-X,9,8,15,-10.7,Intermediate I,Tools,Flesh-rubber,"Light red paste, plentifully mixed with lime ...",Artefacts,Craft,68.136864,27.327120,Making / Craft / Food
3,055-001-X,9,4,7,-6.5,Late II,Pottery - Plain and Banded,NaN,NaN,Artefacts,Domestic,68.136960,27.326984,Feeding / holding
4,055-003-X,9,4,7,-4.6,Late Ib,Pottery - Plain and Banded,NaN,NaN,Artefacts,Domestic,68.136960,27.326984,Feeding / holding


In [ ]:
df_2 = df.join( new_images.set_index('ImageName'), on='Plate', how='left')
df_2['Path'] = df_2['Path'].str.split("./sorted/").str[-1]  # Keep only the relative path

raw_path = "https://raw.githubusercontent.com/LIAVH-MLab/mohenjo-daro/refs/heads/master/"
df_2['Path'] = df_2['Path'].apply(lambda x: f"{raw_path}/{x}" if not pd.isna(x) else None)
df_2

# sample: 
# https://raw.githubusercontent.com/LIAVH-MLab/mohenjo-daro/refs/heads/master//Block_7/House_4/Room_50/037-00b-x.png								

,Plate,Block,House,Room,Level_ft,Time_Cat,Feature,Type,Text,N1,Class,x,y,Class_new,Path
0,054-005-X,9,10,55,-9.1,Late III,Pottery - Incised and Painted - Upper,NaN,NaN,Artefacts,Domestic,68.137116,27.327116,Feeding / holding,https://raw.githubusercontent.com/LIAVH-MLab/m...
1,054-012-X,7,4,72,-7.7,Late III,Tools,Candle-stick,Pottery candle-stick. Made of the usual pink w...,Artefacts,Craft,68.137494,27.326809,Making / Craft / Food,https://raw.githubusercontent.com/LIAVH-MLab/m...
2,054-014-X,9,8,15,-10.7,Intermediate I,Tools,Flesh-rubber,"Light red paste, plentifully mixed with lime ...",Artefacts,Craft,68.136864,27.327120,Making / Craft / Food,https://raw.githubusercontent.com/LIAVH-MLab/m...
3,055-001-X,9,4,7,-6.5,Late II,Pottery - Plain and Banded,NaN,NaN,Artefacts,Domestic,68.136960,27.326984,Feeding / holding,https://raw.githubusercontent.com/LIAVH-MLab/m...
4,055-003-X,9,4,7,-4.6,Late Ib,Pottery - Plain and Banded,NaN,NaN,Artefacts,Domestic,68.136960,27.326984,Feeding / holding,https://raw.githubusercontent.com/LIAVH-MLab/m...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1261,NaN,10,2,NaN,-7.0,Late II and I,Aperture,NaN,Deep hollows here and there in the pavement wh...,Water Features,Drainage,68.136968,27.326793,Drainage,None
1262,NaN,10,2,NaN,-7.0,Late II and I,Pit,NaN,Deep hollows here and there in the pavement wh...,Water Features,Drainage,68.136968,27.326793,Drainage,None
1263,NaN,10,2,NaN,-7.0,Late II and I,Niche,NaN,Deep hollows here and there in the pavement wh...,Water Features,Drainage,68.136968,27.326793,Drainage,None
1264,NaN,7,9,37,-15.5,NaN,Pavement,NaN,"The exact position of the doorways, however, a...",Floor Features,Water Platforms,68.137114,27.326809,Water Platforms,None


In [81]:
df_2[ df_2['Path'].notnull()].sample(1)['Path'].values[0]

'https://raw.githubusercontent.com/LIAVH-MLab/mohenjo-daro/refs/heads/master//Block_7/House_4/Room_72/056-035-x.png'

In [82]:
df_2.to_csv( '/mnt/c/Git_Repo/mohenjo-daro/data/20250407_MJD_processed_data.csv', index=False )